# Tutorial 2: groups and dimensionality reduction

This notebook starts from the same loading workflow as tutorial 1 and adds the next layer of the library:

- grouping samples with `DataManager.load_groups()`
- creating an `ExperimentAnalysis` object
- using dimensionality-reduction plots to explore the dataset

This is the notebook equivalent of the group definition and exploration step in the Metabis server.

## Before you run it

The notebook is self-contained, so it repeats the model and data loading setup. The same bundled Th17 Compass dataset is used throughout the tutorial series.

In [1]:
import sys
from pathlib import Path

from bokeh.io import output_notebook


def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]

    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate

    raise FileNotFoundError("Could not locate the packaged Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import (
    DataManager,
    DataManagerMode,
    ExperimentAnalysis,
    ExpressionData,
    MetabolicModel,
    ReactionData,
    plot_reduced_dimension,
 )
from metabis.version import __version__ as metabis_version

output_notebook()

/inco/home01/g/gdufort/.conda/envs/metabis_test/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
mkdir -p failed for path /home/gdufort/.cache/matplotlib: [Errno 13] Permission denied: '/home/gdufort'
Matplotlib created a temporary cache directory at /tmp/matplotlib-zt0n6ec1 because there was an issue with the default path (/home/gdufort/.cache/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Loading BokehJS ...

In [2]:
def find_metabis_repo_root(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]

    for candidate in [*direct_candidates, *nested_candidates]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "demo").exists()
            and (candidate / "src" / "metabis").exists()
        ):
            return candidate

    raise FileNotFoundError("Could not locate the packaged Metabis repository root.")


def resolve_example_data_dir(repo_root: Path) -> Path:
    data_dir = repo_root / "data"

    if (
        (data_dir / "Th17_Compass_reactions.csv").exists()
        and (data_dir / "Th17_Compass_expression.csv").exists()
    ):
        return data_dir

    raise FileNotFoundError(
        "Could not find the bundled Th17 Compass example files in metabis/data."
    )


repo_root = find_metabis_repo_root(Path.cwd().resolve())
data_dir = resolve_example_data_dir(repo_root)
reaction_data_path = data_dir / "Th17_Compass_reactions.csv"
expression_data_path = data_dir / "Th17_Compass_expression.csv"
analysis_group_1 = "Th17n"
analysis_group_2 = "Th17p"

reaction_loading_options = {
    "data_mode": "dual",
    "reaction_name_format": "rxnRecon3DID",
    "csv_separator": "\t",
    "samples_in": "columns",
    "check_old_identifiers": True,
}

expression_loading_options = {
    "gene_format": "geneSymbols",
    "normalization_options": ["CPM"],
    "data_format": "csv",
    "csv_separator": "\t",
    "samples_in": "columns",
}

model = MetabolicModel(use_cache=True)
alignment_file_path = model.get_alignment_file_path("RECON2")
model.apply_alignment_corrections(alignment_file_path)

reaction_data, _ = ReactionData.create_with_validation(
    str(reaction_data_path),
    metabolic_model=model,
    **reaction_loading_options,
)
expression_data, _ = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    **expression_loading_options,
)

dm = DataManager(model, reaction_data, expression_data, None)

print(f"Metabis version: {metabis_version}")
print(f"Loaded {len(reaction_data.get_sample_ids())} samples")

Metabis version: 0.0.1
Loaded 290 samples


## 1. Define groups

The `DataManager` owns grouping because groups become shared state for later analysis objects. In this example, `sample_prefix` is enough because the sample names already start with `Th17n_...` or `Th17p_...`.

In [3]:
grouping_result = dm.load_groups(criteria="sample_prefix")
group_ids = sorted(dm.get_group_ids())
sample_counts = {group_id: len(dm.get_group_samples(group_id)) for group_id in group_ids}

print(f"Detected groups: {group_ids}")
print(f"Samples per group: {sample_counts}")
print(f"Group colors: {dm.get_group_colors()}")
assert analysis_group_1 in group_ids and analysis_group_2 in group_ids
grouping_result

Detected groups: ['Th17n', 'Th17p']
Samples per group: {'Th17n': 151, 'Th17p': 139}
Group colors: {'Th17p': '#008080', 'Th17n': '#FF8C00'}


{'status': 'success', 'messages': ['Created 2 groups']}

## 2. Create the experiment analysis object

`ExperimentAnalysis` is the library object that summarizes activity patterns within the defined groups and stores dimensionality-reduction results.

The public pattern is:

1. create `ExperimentAnalysis(dm)`
2. run or extend the analysis
3. register it back into the `DataManager`
4. switch the `DataManager` to `EXPERIMENT_ANALYSIS` mode

In [4]:
analysis = ExperimentAnalysis(dm)
analysis.run_analysis(
    include_pca=True,
    pca_data_types=["reactions", "expression"],
    n_components=10,
)

analysis.calculate_tsne(
    data_type="reactions",
    perplexity=10.0,
    n_variable_features=3000,
)
analysis.calculate_umap(
    data_type="reactions",
    n_neighbors=10,
    min_dist=0.2,
    n_variable_features=3000,
)
analysis.calculate_pca(
    data_type="expression",
    n_components=6,
    n_variable_features=3000,
)

dm.update_experiment_analysis(analysis)
dm.set_mode(DataManagerMode.EXPERIMENT_ANALYSIS)
dm.set_analysis_group(analysis_group_1)

print(f"Current mode: {dm.get_mode()}")
print(f"PCA result types: {list(analysis.pca_results.keys())}")
print(f"t-SNE result types: {list(analysis.tsne_results.keys())}")
print(f"UMAP result types: {list(analysis.umap_results.keys())}")

/inco/home01/g/gdufort/.conda/envs/metabis_test/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Current mode: DataManagerMode.EXPERIMENT_ANALYSIS
PCA result types: ['reactions', 'expression']
t-SNE result types: ['reactions']
UMAP result types: ['reactions']


## 3. PCA in reaction space

`plot_reduced_dimension()` is the high-level plotting helper. It reads the stored reduction results from the `ExperimentAnalysis` object attached to the `DataManager`.

In [5]:
plot_reduced_dimension(
    dm,
    method="PCA",
    data_type="reactions",
    figsize=(900, 650),
    n_variable_features=3000,
    return_plot=False,
)

## 4. t-SNE in reaction space

The same plotting entry point works for other reductions once the `ExperimentAnalysis` object has calculated them.

In [6]:
plot_reduced_dimension(
    dm,
    method="T-SNE",
    data_type="reactions",
    figsize=(900, 650),
    n_variable_features=3000,
    return_plot=False,
)

## 5. UMAP in reaction space

UMAP is often a useful complementary view when you want a more local neighborhood-based embedding of the same samples.

In [7]:
plot_reduced_dimension(
    dm,
    method="UMAP",
    data_type="reactions",
    figsize=(900, 650),
    n_variable_features=3000,
    return_plot=False,
)

## 6. The same API also works on expression data

The `data_type` argument lets you switch between the reaction matrix and the expression matrix without changing the overall workflow.

In [8]:
plot_reduced_dimension(
    dm,
    method="PCA",
    data_type="expression",
    figsize=(900, 650),
    n_variable_features=3000,
    return_plot=False,
)

## Takeaways

At this stage you have introduced the first stateful analysis object in the library: `ExperimentAnalysis`.

The next notebook switches from exploratory group-aware views to direct group comparison, significance thresholds, and statistical plots.